In [0]:
%sql 
create volume if not exists ml_lab

In [0]:
catalog_name=spark.sql('select current_catalog()').collect()[0][0]

In [0]:
df=spark.read.format('csv').options(inferSchema='true', header='true').load('/Volumes/dbws01/default/ml_lab/diabetes.csv')

In [0]:
df.columns

['Pregnancies',
 'Glucose',
 'BloodPressure',
 'SkinThickness',
 'Insulin',
 'BMI',
 'DiabetesPedigreeFunction',
 'Age',
 'Outcome']

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

schema=StructType([
    StructField('Pregnancies', IntegerType(), True),
    StructField('Glucose', IntegerType(), True),
    StructField('BloodPressure', IntegerType(), True),
    StructField('SkinThickness', IntegerType(), True),
    StructField('DiabetesPedigreeFunction', FloatType()),
    StructField('Age',IntegerType()),
    StructField('Outcome', IntegerType())
])

df=spark.read.format('csv').options(inferSchema='true', header='true', schema=schema).load('/Volumes/dbws01/default/ml_lab/diabetes.csv').dropna()


In [0]:
splits=df.randomSplit([0.7,0.3])
train=splits[0]
test=splits[1]
print(train.count(), test.count())

533 235


In [0]:
#normalize
from pyspark.ml.feature import VectorAssembler, MinMaxScaler
numerical_features=['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
numericColVector=VectorAssembler(inputCols=numerical_features, outputCol="numericFeatures")

vectorizedData=numericColVector.transform(df) 


In [0]:
minMax=MinMaxScaler(inputCol=numericColVector.getOutputCol(), outputCol="normalizedFeatures")
scaled_data=minMax.fit(vectorizedData).transform(vectorizedData)

In [0]:
prepped_data=scaled_data[col('normalizedFeatures'), col('Outcome')]
display(prepped_data)

normalizedFeatures,Outcome
"Map(vectorType -> dense, length -> 8, values -> List(0.3529411764705882, 0.7437185929648241, 0.5901639344262295, 0.3535353535353536, 0.0, 0.5007451564828614, 0.23441502988898377, 0.48333333333333334))",1
"Map(vectorType -> dense, length -> 8, values -> List(0.058823529411764705, 0.4271356783919598, 0.5409836065573771, 0.29292929292929293, 0.0, 0.3964232488822653, 0.11656703672075147, 0.16666666666666666))",0
"Map(vectorType -> dense, length -> 8, values -> List(0.47058823529411764, 0.9195979899497487, 0.5245901639344263, 0.0, 0.0, 0.34724292101341286, 0.25362937660119556, 0.18333333333333332))",1
"Map(vectorType -> dense, length -> 8, values -> List(0.058823529411764705, 0.4472361809045226, 0.5409836065573771, 0.23232323232323235, 0.1111111111111111, 0.41877794336810736, 0.038001707941929974, 0.0))",0
"Map(vectorType -> dense, length -> 8, values -> List(0.0, 0.6884422110552764, 0.3278688524590164, 0.3535353535353536, 0.19858156028368795, 0.6423248882265277, 0.9436379163108454, 0.2))",1
"Map(vectorType -> dense, length -> 8, values -> List(0.29411764705882354, 0.5829145728643216, 0.6065573770491803, 0.0, 0.0, 0.3815201192250373, 0.052519214346712216, 0.15))",0
"Map(vectorType -> dense, length -> 8, values -> List(0.1764705882352941, 0.3919597989949749, 0.4098360655737705, 0.32323232323232326, 0.10401891252955082, 0.4619970193740686, 0.07258753202391117, 0.08333333333333333))",1
"Map(vectorType -> dense, length -> 8, values -> List(0.5882352941176471, 0.577889447236181, 0.0, 0.0, 0.0, 0.526080476900149, 0.023911187019641334, 0.13333333333333333))",0
"Map(vectorType -> dense, length -> 8, values -> List(0.11764705882352941, 0.9899497487437187, 0.5737704918032788, 0.4545454545454546, 0.6418439716312057, 0.4545454545454546, 0.034158838599487616, 0.5333333333333333))",1
"Map(vectorType -> dense, length -> 8, values -> List(0.47058823529411764, 0.628140703517588, 0.7868852459016393, 0.0, 0.0, 0.0, 0.06575576430401367, 0.55))",1


In [0]:
from pyspark.ml.classification import LogisticRegression
lr=LogisticRegression(labelCol='Outcome', featuresCol='normalizedFeatures')
model=lr.fit(prepped_data)

In [0]:
#test
vectorizedTestData=numericColVector.transform(test)
scaledTestData=minMax.fit(vectorizedTestData).transform(vectorizedTestData)
preppedTestData=scaledTestData[col("normalizedFeatures"), col('Outcome')]

prediction=model.transform(preppedTestData)
predicted=prediction.select(col('normalizedFeatures'), col('probability'),col('prediction'),'Outcome')
display(predicted)

normalizedFeatures,probability,prediction,Outcome
"Map(vectorType -> dense, length -> 8, values -> List(0.0, 0.3724489795918367, 0.0, 0.0, 0.0, 0.31445603576751124, 0.11505229649840837, 0.06666666666666667))","Map(vectorType -> dense, length -> 2, values -> List(0.9610457268898309, 0.03895427311016908))",0.0,0
"Map(vectorType -> dense, length -> 8, values -> List(0.0, 0.37755102040816324, 0.45614035087719296, 0.17857142857142855, 0.052941176470588235, 0.41430700447093893, 0.08185538881309688, 0.016666666666666666))","Map(vectorType -> dense, length -> 2, values -> List(0.9698028303793579, 0.030197169620642117))",0.0,0
"Map(vectorType -> dense, length -> 8, values -> List(0.0, 0.39795918367346933, 0.7719298245614035, 0.5178571428571428, 0.058823529411764705, 0.549925484351714, 0.15688949522510232, 0.0))","Map(vectorType -> dense, length -> 2, values -> List(0.9456485569887622, 0.05435144301123784))",0.0,0
"Map(vectorType -> dense, length -> 8, values -> List(0.0, 0.42857142857142855, 0.7192982456140351, 0.5535714285714285, 0.1838235294117647, 0.5692995529061103, 0.0654843110504775, 0.03333333333333333))","Map(vectorType -> dense, length -> 2, values -> List(0.9391541842342613, 0.060845815765738664))",0.0,0
"Map(vectorType -> dense, length -> 8, values -> List(0.0, 0.4744897959183673, 0.5263157894736842, 0.0, 0.0, 0.526080476900149, 0.07912687585266032, 0.06666666666666667))","Map(vectorType -> dense, length -> 2, values -> List(0.8957696392680681, 0.10423036073193193))",0.0,0
"Map(vectorType -> dense, length -> 8, values -> List(0.0, 0.4744897959183673, 0.5263157894736842, 0.4464285714285714, 0.13529411764705881, 0.42771982116244417, 0.20145520691223287, 0.016666666666666666))","Map(vectorType -> dense, length -> 2, values -> List(0.9325144879323708, 0.0674855120676292))",0.0,0
"Map(vectorType -> sparse, length -> 8, indices -> List(1, 6, 7), values -> List(0.47959183673469385, 0.07594361073215099, 0.06666666666666667))","Map(vectorType -> dense, length -> 2, values -> List(0.9882856336588959, 0.011714366341104121))",0.0,0
"Map(vectorType -> dense, length -> 8, values -> List(0.0, 0.4846938775510204, 0.7017543859649122, 0.8035714285714285, 0.13529411764705881, 0.5439642324888228, 0.10959527057753526, 0.08333333333333333))","Map(vectorType -> dense, length -> 2, values -> List(0.9056339449300955, 0.09436605506990448))",0.0,0
"Map(vectorType -> dense, length -> 8, values -> List(0.0, 0.5051020408163265, 0.0, 0.0, 0.0, 0.3725782414307005, 0.0745793542519327, 0.016666666666666666))","Map(vectorType -> dense, length -> 2, values -> List(0.8871293022040473, 0.11287069779595271))",0.0,0
"Map(vectorType -> dense, length -> 8, values -> List(0.0, 0.5102040816326531, 0.6140350877192982, 0.46428571428571425, 0.07352941176470588, 0.459016393442623, 0.23101409731696226, 0.0))","Map(vectorType -> dense, length -> 2, values -> List(0.9016811441876139, 0.09831885581238609))",0.0,0


In [0]:
#evaluate
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
evaluator=MulticlassClassificationEvaluator(labelCol='Outcome', predictionCol='prediction', metricName='accuracy')

accuracy=evaluator.evaluate(prediction)
print(accuracy)

0.7659574468085106


In [0]:
evaluator=MulticlassClassificationEvaluator(labelCol='Outcome', predictionCol='prediction', metricName='weightedPrecision')
precision=evaluator.evaluate(prediction)
print(precision)

0.7609141468742686


In [0]:
evaluator=MulticlassClassificationEvaluator(labelCol='Outcome', predictionCol='prediction', metricName='weightedFMeasure')
F1score=evaluator.evaluate(prediction)
print(F1score)

0.7600220570139593


In [0]:
# as a pipeline
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, MinMaxScaler

numFeatures=['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

numVector=VectorAssembler(inputCols=numFeatures, outputCol='numericFeatures')
numScaler=MinMaxScaler(inputCol=numVector.getOutputCol(), outputCol='normalizedFeatures')
featureVector=VectorAssembler(inputCols=['normalizedFeatures'], outputCol="features")
algo=LogisticRegression(labelCol='Outcome', featuresCol='features')

pipeline=Pipeline(stages=[numVector, numScaler, featureVector, algo])

model=pipeline.fit(train)



In [0]:
prediction=model.transform(test)
predicted=prediction.select(col('normalizedFeatures'), col('probability'),col('prediction'),'Outcome')
display(predicted)


normalizedFeatures,probability,prediction,Outcome
"Map(vectorType -> dense, length -> 8, values -> List(0.0, 0.36683417085427134, 0.0, 0.0, 0.0, 0.3552188552188552, 0.11272416737830913, 0.0784313725490196))","Map(vectorType -> dense, length -> 2, values -> List(0.9659140230704605, 0.03408597692953952))",0.0,0
"Map(vectorType -> dense, length -> 8, values -> List(0.0, 0.37185929648241206, 0.42622950819672134, 0.10101010101010102, 0.0425531914893617, 0.468013468013468, 0.08155422715627668, 0.0196078431372549))","Map(vectorType -> dense, length -> 2, values -> List(0.9680535949368013, 0.03194640506319868))",0.0,0
"Map(vectorType -> dense, length -> 8, values -> List(0.0, 0.3919597989949749, 0.7213114754098361, 0.29292929292929293, 0.04728132387706856, 0.6212121212121212, 0.15200683176771987, 0.0))","Map(vectorType -> dense, length -> 2, values -> List(0.9428683261930787, 0.0571316738069213))",0.0,0
"Map(vectorType -> dense, length -> 8, values -> List(0.0, 0.4221105527638191, 0.6721311475409837, 0.31313131313131315, 0.14775413711583923, 0.6430976430976432, 0.06618274978650726, 0.0392156862745098))","Map(vectorType -> dense, length -> 2, values -> List(0.92880849846444, 0.07119150153555998))",0.0,0
"Map(vectorType -> dense, length -> 8, values -> List(0.0, 0.46733668341708545, 0.49180327868852464, 0.0, 0.0, 0.5942760942760942, 0.07899231426131512, 0.0784313725490196))","Map(vectorType -> dense, length -> 2, values -> List(0.8823962944705009, 0.11760370552949906))",0.0,0
"Map(vectorType -> dense, length -> 8, values -> List(0.0, 0.46733668341708545, 0.49180327868852464, 0.25252525252525254, 0.10874704491725767, 0.48316498316498313, 0.19385140905209222, 0.0196078431372549))","Map(vectorType -> dense, length -> 2, values -> List(0.9293314068819929, 0.0706685931180071))",0.0,0
"Map(vectorType -> sparse, length -> 8, indices -> List(1, 6, 7), values -> List(0.4723618090452261, 0.07600341588385993, 0.0784313725490196))","Map(vectorType -> dense, length -> 2, values -> List(0.9914440998828392, 0.008555900117160786))",0.0,0
"Map(vectorType -> dense, length -> 8, values -> List(0.0, 0.47738693467336685, 0.6557377049180328, 0.4545454545454546, 0.10874704491725767, 0.6144781144781145, 0.107600341588386, 0.09803921568627451))","Map(vectorType -> dense, length -> 2, values -> List(0.9141056049754169, 0.08589439502458307))",0.0,0
"Map(vectorType -> dense, length -> 8, values -> List(0.0, 0.49748743718592964, 0.0, 0.0, 0.0, 0.42087542087542085, 0.07472245943637916, 0.0196078431372549))","Map(vectorType -> dense, length -> 2, values -> List(0.8979165733851174, 0.10208342661488257))",0.0,0
"Map(vectorType -> dense, length -> 8, values -> List(0.0, 0.5025125628140703, 0.5737704918032788, 0.26262626262626265, 0.0591016548463357, 0.5185185185185185, 0.22160546541417592, 0.0))","Map(vectorType -> dense, length -> 2, values -> List(0.9014354217343862, 0.09856457826561382))",0.0,0


In [0]:
#saving the model 
model.save('/Volumes/dbws01/default/ml_lab/diabetes.model')


In [0]:
#load the model
from pyspark.ml.pipeline import PipelineModel
persistedModel=PipelineModel.load('/Volumes/dbws01/default/ml_lab/diabetes.model')

rawData=spark.createDataFrame(
[ {'Pregnancies':8, 'Glucose':183, 'BloodPressure':64, 'SkinThickness':0, 'Insulin':0, 'BMI':40.3, 'DiabetesPedigreeFunction':0.627,'Age':34}
]
)

predictions=persistedModel.transform(rawData)


In [0]:
display(predictions)

Age,BMI,BloodPressure,DiabetesPedigreeFunction,Glucose,Insulin,Pregnancies,SkinThickness,numericFeatures,normalizedFeatures,features,rawPrediction,probability,prediction
34,40.3,64,0.627,183,0,8,0,"Map(vectorType -> dense, length -> 8, values -> List(8.0, 183.0, 64.0, 0.0, 0.0, 40.3, 0.627, 34.0))","Map(vectorType -> dense, length -> 8, values -> List(0.47058823529411764, 0.9195979899497487, 0.5245901639344263, 0.0, 0.0, 0.6784511784511784, 0.23441502988898377, 0.2549019607843137))","Map(vectorType -> dense, length -> 8, values -> List(0.47058823529411764, 0.9195979899497487, 0.5245901639344263, 0.0, 0.0, 0.6784511784511784, 0.23441502988898377, 0.2549019607843137))","Map(vectorType -> dense, length -> 2, values -> List(-3.0013237607820997, 3.0013237607820997))","Map(vectorType -> dense, length -> 2, values -> List(0.04736610590244382, 0.9526338940975562))",1.0


In [0]:
display(predictions.select(col('prediction')))

prediction
1.0
